# محرك الصور الاحترافي المجاني: ComfyUI (SDXL + ControlNet + IP-Adapter)

هذا الدفتر يشغّل **خادم صور سينمائي** على GPU جوجل المجانية (Colab) أو كاجل (Kaggle):

- **SDXL** بتوليد عالي الجودة (1920×1080).
- **ControlNet OpenPose**: يحافظ على وضعية الشخصية في كل لقطة.
- **IP-Adapter**: يثبّت ملامح الشخصية وملابسها عبر صورة مرجعية (شيت الشخصية).

التكلفة: **صفر**. بعد التشغيل ستحصل على رابط (مثل `https://xxx.trycloudflare.com`) تنسخه إلى حقل **API Base** في إعدادات موقعك، وتختار `IMAGE_PROVIDER=comfy` في الخادم.

الطريقة: `Runtime → Change runtime type → T4 GPU` ثم `Runtime → Run all`.

In [ ]:
# 1) تثبيت وتشغيل ComfyUI
import os, subprocess, sys
!apt-get -qq update > /dev/null && apt-get -qq install -y git aria2 > /dev/null 2>&1 || true
if not os.path.isdir('/content/ComfyUI'):
    os.system('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git')
%cd /content/ComfyUI
os.system('pip install -q -r requirements.txt')
# العقد المخصصة المطلوبة (IP-Adapter و ControlNet المتقدم)
os.makedirs('custom_nodes', exist_ok=True)
if not os.path.isdir('custom_nodes/ComfyUI_IPAdapter_plus'):
    os.system('git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git custom_nodes/ComfyUI_IPAdapter_plus')
if not os.path.isdir('custom_nodes/comfyui_controlnet_aux'):
    os.system('git clone --depth 1 https://github.com/Fannovel16/comfyui_controlnet_aux.git custom_nodes/comfyui_controlnet_aux')
os.system('pip install -q -r custom_nodes/ComfyUI_IPAdapter_plus/requirements.txt')
os.system('pip install -q -r custom_nodes/comfyui_controlnet_aux/requirements.txt')
print('ComfyUI ready')

In [ ]:
#@title 2) تنزيل النماذج المجانية (عدّل الروابط حسب رغبتك)
#@markdown موديل SDXL أساسي من HuggingFace (مجاني تمامًا):
SDXL_URL = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors" #@param {type:"string"}
#@markdown ControlNet OpenPose المخصص لـ SDXL:
OPENPOSE_URL = "https://huggingface.co/thibaud/controlnet-openpose-sdxl-1.0/resolve/main/control_openpose-sdxl-1.0.safetensors" #@param {type:"string"}
#@markdown IP-Adapter Face لـ SDXL (لثبات الوجه والملابس):
IPADAPTER_URL = "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus-face_sdxl_vit-h.safetensors" #@param {type:"string"}
IPADAPTER_MODEL = "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors" #@param {type:"string"}

import os
import urllib.request

def download(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 10**7:
        print('موجود:', dest); return
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print('تحميل:', url.split('/')[-1])
    os.system(f'aria2c -x4 -s4 -q -c "{url}" -d "{os.path.dirname(dest)}" -o "{os.path.basename(dest)}"')
    if not (os.path.exists(dest) and os.path.getsize(dest) > 10**7):
        os.system(f'wget -q -O "{dest}" "{url}"')

download(SDXL_URL, 'models/checkpoints/sd_xl_base_1.0.safetensors')
download(OPENPOSE_URL, 'models/controlnet/control_openpose-sdxl-1.0.safetensors')
download(IPADAPTER_URL, 'models/ipadapter/ip-adapter-plus-face_sdxl_vit-h.safetensors')
# صورة التشفير (CLIP vision) — باسم يطابق ما يطلبه IPAdapter (PLUS FACE):
download(IPADAPTER_MODEL, 'models/clip_vision/ViT-H-14-s32B-b79K.safetensors')
print('النماذج جاهزة')
print('  checkpoint:', os.path.getsize('models/checkpoints/sd_xl_base_1.0.safetensors')//10**6, 'MB')
print('  controlnet:', os.path.getsize('models/controlnet/control_openpose-sdxl-1.0.safetensors')//10**6, 'MB')
print('  ipadapter:', os.path.getsize('models/ipadapter/ip-adapter-plus-face_sdxl_vit-h.safetensors')//10**6, 'MB')
print('  clip vision:', os.path.getsize('models/clip_vision/ViT-H-14-s32B-b79K.safetensors')//10**6, 'MB')
print('\nتلميح: لو حطيت موديل كرتوني بأي اسم فيه toon/cartoon/comic في models/checkpoints،')
print('الخادم سيختاره تلقائيًا بدل الأساسي.')

In [ ]:
# 3) تشغيل خادم ComfyUI في الخلفية + نفق مجاني (cloudflared)
import os, subprocess, time, shutil

with open('run.log', 'w') as f:
    p = subprocess.Popen([sys.executable, 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
                         stdout=f, stderr=subprocess.STDOUT)
print('ComfyUI pid:', p.pid)

# انتظار جاهزية الخادم
import urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        print('ComfyUI يعمل على http://127.0.0.1:8188')
        break
    except Exception:
        time.sleep(2)
else:
    print('تعذر الوصول لـ ComfyUI — راجع run.log')
    print(open('run.log').read()[-1500:])

if not shutil.which('cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared')
if shutil.which('cloudflared') or os.path.exists('./cloudflared'):
    import subprocess as sp
    cf = './cloudflared' if os.path.exists('./cloudflared') else 'cloudflared'
    with open('tunnel.log', 'w') as f:
        tp = sp.Popen([cf, 'tunnel', '--url', 'http://127.0.0.1:8188'], stdout=f, stderr=subprocess.STDOUT)
    for i in range(30):
        txt = open('tunnel.log').read()
        if 'trycloudflare.com' in txt:
            url = [w for w in txt.split() if 'trycloudflare.com' in w][0]
            break
        time.sleep(2)
    else:
        url = None
    if url:
        print('\nنفّذ الرابط التالي في الجهاز لضبط COMFY_URL:')
        print('  export COMFY_URL=' + url)
        print('\nفي إعدادات موقعك ضع API Base = ' + url)
        print('\n**اترك هذا الدفتر مفتوحًا أثناء الإنتاج** (النفق يموت عند إغلاق الجلسة).')
    else:
        print('لم يظهر رابط النفق بعد — اقرأ tunnel.log:')
        print(open('tunnel.log').read()[-800:])
else:
    print('تثبيت cloudflared فشل — استخدم رابط المنفذ العام من Colab (Web → port 8188)')